<a href="https://colab.research.google.com/github/dee431/Project-FORESIGHT-AI-Powered-Demand-Inventory-Intelligence-Platform/blob/main/Project_FORESIGHT_%E2%80%93_AI_Powered_Demand_%26_Inventory_Intelligence_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cell 1: Install & Import Dependencies**

In [1]:
# Cell 1: Load essential analytics, machine learning, and visualization libraries
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✅ Dependencies loaded successfully.")

✅ Dependencies loaded successfully.


## **Cell 2: Synthetic E-Commerce Inventory & Demand Dataset Generation**

In [2]:
# Cell 2: Generate realistic 1-year historical demand and inventory data for 3 SKUs
np.random.seed(42)

def generate_foresight_data():
    dates = pd.date_range(start="2025-01-01", end="2025-12-31", freq="D")
    skus = [
        {"sku_id": "SKU-101", "name": "Wireless Headphones", "base_demand": 50, "unit_cost": 45, "price": 99, "lead_time_days": 5},
        {"sku_id": "SKU-202", "name": "Mechanical Keyboard", "base_demand": 30, "unit_cost": 60, "price": 120, "lead_time_days": 7},
        {"sku_id": "SKU-303", "name": "Ergonomic Chair", "base_demand": 15, "unit_cost": 150, "price": 350, "lead_time_days": 10}
    ]

    df_list = []

    for sku in skus:
        n_days = len(dates)
        # Add weekly seasonality + trend + random variance
        day_of_week_factor = 1 + 0.3 * np.sin(2 * np.pi * dates.dayofweek / 7)
        month_factor = 1 + 0.2 * np.cos(2 * np.pi * dates.month / 12)
        noise = np.random.normal(1, 0.15, n_days)

        daily_sales = np.maximum(0, (sku["base_demand"] * day_of_week_factor * month_factor * noise)).astype(int)

        # Simulate stock levels
        starting_stock = sku["base_demand"] * 15
        stock_levels = []
        current_stock = starting_stock

        for sales in daily_sales:
            current_stock = max(0, current_stock - sales)
            stock_levels.append(current_stock)
            # Replenish stock when depleted
            if current_stock < (sku["base_demand"] * sku["lead_time_days"]):
                current_stock += sku["base_demand"] * 12

        df_sku = pd.DataFrame({
            "date": dates,
            "sku_id": sku["sku_id"],
            "sku_name": sku["name"],
            "daily_sales": daily_sales,
            "stock_on_hand": stock_levels,
            "unit_cost": sku["unit_cost"],
            "unit_price": sku["price"],
            "lead_time_days": sku["lead_time_days"]
        })
        df_list.append(df_sku)

    return pd.concat(df_list, ignore_index=True)

df_raw = generate_foresight_data()
print(f"✅ Generated dataset with {len(df_raw)} records across {df_raw['sku_id'].nunique()} SKUs.")
df_raw.head()

✅ Generated dataset with 1095 records across 3 SKUs.


,date,sku_id,sku_name,daily_sales,stock_on_hand,unit_cost,unit_price,lead_time_days
0,2025-01-01,SKU-101,Wireless Headphones,81,669,45,99,5
1,2025-01-02,SKU-101,Wireless Headphones,64,605,45,99,5
2,2025-01-03,SKU-101,Wireless Headphones,55,550,45,99,5
3,2025-01-04,SKU-101,Wireless Headphones,50,500,45,99,5
4,2025-01-05,SKU-101,Wireless Headphones,43,457,45,99,5


# **Cell 3: Feature Engineering Pipeline**

In [3]:
# Cell 3: Create time-series lag and rolling statistics features
def create_features(df):
    data = df.copy().sort_values(["sku_id", "date"])

    # Temporal features
    data["day_of_week"] = data["date"].dt.dayofweek
    data["month"] = data["date"].dt.month
    data["is_weekend"] = data["day_of_week"].isin([5, 6]).astype(int)

    # Lag features
    for lag in [1, 7, 14, 30]:
        data[f"sales_lag_{lag}"] = data.groupby("sku_id")["daily_sales"].shift(lag)

    # Rolling averages & standard deviations
    for window in [7, 14, 30]:
        data[f"rolling_mean_{window}"] = data.groupby("sku_id")["daily_sales"].transform(lambda x: x.shift(1).rolling(window).mean())
        data[f"rolling_std_{window}"] = data.groupby("sku_id")["daily_sales"].transform(lambda x: x.shift(1).rolling(window).std())

    return data.dropna()

df_featured = create_features(df_raw)
print("✅ Feature engineering completed.")

✅ Feature engineering completed.


# **Cell 4: Train Demand Forecasting Model**

In [4]:
# Cell 4: Train Random Forest Regressor and evaluate SKU-level demand predictions
feature_cols = [
    "day_of_week", "month", "is_weekend",
    "sales_lag_1", "sales_lag_7", "sales_lag_14", "sales_lag_30",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_30",
    "rolling_std_7", "rolling_std_14", "rolling_std_30"
]

train_data = df_featured[df_featured["date"] < "2025-11-01"]
test_data = df_featured[df_featured["date"] >= "2025-11-01"].copy()

X_train, y_train = train_data[feature_cols], train_data["daily_sales"]
X_test, y_test = test_data[feature_cols], test_data["daily_sales"]

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

test_data["predicted_sales"] = model.predict(X_test).round().astype(int)

mae = mean_absolute_error(y_test, test_data["predicted_sales"])
rmse = root_mean_squared_error(y_test, test_data["predicted_sales"])

print(f"✅ Demand Model Trained.")
print(f"📊 Overall Model MAE: {mae:.2f} units | RMSE: {rmse:.2f} units")

✅ Demand Model Trained.
📊 Overall Model MAE: 5.15 units | RMSE: 7.11 units


# **Cell 5: Inventory Intelligence Engine (Safety Stock, ROP, Risk Detection)**

In [5]:
# Cell 5: Calculate Safety Stock, Reorder Point (ROP), and risk classifications
def calculate_inventory_metrics(data):
    # Service factor Z for 95% service level
    Z = 1.65

    # Calculate daily sales standard deviation per SKU
    sku_stats = data.groupby("sku_id").agg(
        avg_predicted_demand=("predicted_sales", "mean"),
        demand_std=("predicted_sales", "std"),
        lead_time=("lead_time_days", "first"),
        current_stock=("stock_on_hand", "last"),
        unit_cost=("unit_cost", "first"),
        unit_price=("unit_price", "first"),
        sku_name=("sku_name", "first")
    ).reset_index()

    # Safety Stock Formula: Z * std_demand * sqrt(lead_time)
    sku_stats["safety_stock"] = (Z * sku_stats["demand_std"] * np.sqrt(sku_stats["lead_time"])).round().astype(int)

    # Reorder Point (ROP) Formula: (Lead Time * Avg Daily Demand) + Safety Stock
    sku_stats["reorder_point"] = ((sku_stats["avg_predicted_demand"] * sku_stats["lead_time"]) + sku_stats["safety_stock"]).round().astype(int)

    # Optimal Max Stock
    sku_stats["max_target_stock"] = sku_stats["reorder_point"] + (sku_stats["avg_predicted_demand"] * 14)

    # Risk Assessment Logic
    def get_risk(row):
        if row["current_stock"] <= row["reorder_point"]:
            return "🔴 HIGH: Stockout Risk (Reorder Required)"
        elif row["current_stock"] > row["max_target_stock"]:
            return "🟡 WARNING: Overstock Risk"
        else:
            return "🟢 HEALTHY: Optimal Inventory"

    sku_stats["status"] = sku_stats.apply(get_risk, axis=1)

    # Recommended Order Quantity
    sku_stats["recommended_reorder_qty"] = np.where(
        sku_stats["current_stock"] <= sku_stats["reorder_point"],
        sku_stats["max_target_stock"] - sku_stats["current_stock"],
        0
    )

    return sku_stats

inventory_insights = calculate_inventory_metrics(test_data)
print("✅ Inventory Intelligence Calculations Complete.\n")
display(inventory_insights[["sku_id", "sku_name", "current_stock", "reorder_point", "safety_stock", "status", "recommended_reorder_qty"]])

✅ Inventory Intelligence Calculations Complete.



,sku_id,sku_name,current_stock,reorder_point,safety_stock,status,recommended_reorder_qty
0,SKU-101,Wireless Headphones,629,314,37,🟢 HEALTHY: Optimal Inventory,0.0
1,SKU-202,Mechanical Keyboard,525,281,34,🟢 HEALTHY: Optimal Inventory,0.0
2,SKU-303,Ergonomic Chair,241,190,19,🟢 HEALTHY: Optimal Inventory,0.0


# **Cell 6: FORESIGHT Analytics & Intelligence Dashboard**

In [6]:
# Cell 6: Render interactive KPI dashboard and demand forecasting visualizer
selected_sku = "SKU-101"
sku_plot_data = test_data[test_data["sku_id"] == selected_sku]
sku_metrics = inventory_insights[inventory_insights["sku_id"] == selected_sku].iloc[0]

# Create Dashboard Layout
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        f"Demand Forecast vs Actuals ({selected_sku})",
        "Inventory Health Status Across SKUs",
        f"Stock Level vs Reorder Point ({selected_sku})",
        "Recommended Reorder Quantities"
    ),
    specs=[[{"type": "scatter"}, {"type": "pie"}],
           [{"type": "scatter"}, {"type": "bar"}]]
)

# Plot 1: Forecast vs Actual Demand
fig.add_trace(
    go.Scatter(x=sku_plot_data["date"], y=sku_plot_data["daily_sales"], name="Actual Sales", line=dict(color="blue")),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=sku_plot_data["date"], y=sku_plot_data["predicted_sales"], name="Forecasted Sales", line=dict(color="orange", dash="dash")),
    row=1, col=1
)

# Plot 2: Inventory Status Distribution
status_counts = inventory_insights["status"].value_counts()
fig.add_trace(
    go.Pie(labels=status_counts.index, values=status_counts.values, hole=0.4),
    row=1, col=2
)

# Plot 3: Stock vs ROP
fig.add_trace(
    go.Scatter(x=sku_plot_data["date"], y=sku_plot_data["stock_on_hand"], name="Stock Level", line=dict(color="green")),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(
        x=sku_plot_data["date"],
        y=[sku_metrics["reorder_point"]] * len(sku_plot_data),
        name="Reorder Point (ROP)",
        line=dict(color="red", dash="dot")
    ),
    row=2, col=1
)

# Plot 4: Reorder Recommendations
fig.add_trace(
    go.Bar(x=inventory_insights["sku_id"], y=inventory_insights["recommended_reorder_qty"], marker_color="crimson", name="Reorder Qty"),
    row=2, col=2
)

fig.update_layout(height=750, title_text="<b>Project FORESIGHT – Demand & Inventory Intelligence Platform</b>", showlegend=True)
fig.show()